# Pipeline

The pipeline automatically runs the 3 basic processing steps:

- find objects on the first plate, that have no corresponding object on the second plate;
- statistically compare between the objects with no match, with the ones that have a match;
- display the non-matched objects that have a higher likelihood of being a "real event" (and 
  not a plate artifact).
  
The sequence is controlled by an entry in the *sequences* dictionary in file *settings.py*.
These are sequences of plate ID numbers. 

Existing sequences currently in the file were produced
by notebook *footoprints.ipynb*, and include all sequenecs for the **Grosser Schmidt-Spiegel**
telescope that contain plates that overlap on the sky by more than 50%, and were taken in the 
same nigth, or maybe one night before or after. One can place in there any sequence that one 
wants to study though.

Each step of the pipelin is run by a separate notebook, with results dumped in subdirectory 
**./html/**. Results have the same format as the corresponding input notebook run for a 
particular pair of plates, but are read-only, and formatted as an HTML web page. 

These notebooks can be run separately in interactive form. The input for a notebook is controlled 
by the content of file *dataset.json*, which defines the pair of plates one wants to study. This file
is rewritten by the pipeline, so make sure it points to the correct pair of plates before running 
any notebook (pay attention to the spelling, it's a JSON file which has stringent formatting 
requirements).

The pipelne requires that all data be previously installed in the data directory (defined in file
*settings.py*). The data can be automatically downloaded by script *download.ipynb*. This script is
also driven by the sequences in *settings.py*.

In [ ]:
import os
import json
from importlib import reload

import settings
from settings import DATAPATH, sequences
from library import update_dataset

In [ ]:
# output path
output_path = os.path.join(DATAPATH, "html")

In [ ]:
# input sequence
sequence = sequences['seq03']

In [ ]:
for i in range(len((sequence)) - 1):
    
    plate_id_str = str(sequence[i])
    next_plate_id_str = str(sequence[i+1])

    key = plate_id_str + ',' + next_plate_id_str
    print("START processing dataset: ", key)
    
    update_dataset(key)
    reload(settings)
    from settings import current_dataset
    
    suffix = plate_id_str + "_" + next_plate_id_str + ".html"
    
    try:
        filename = os.path.join(output_path, "find_mismatches_" + suffix)
        !jupyter nbconvert --to html --execute find_mismatches.ipynb --output $filename

        filename = os.path.join(output_path, "psf_analysis_" + suffix)
        !jupyter nbconvert --to html --execute psf_analysis.ipynb --output $filename

        filename = os.path.join(output_path, "display_nomatches_" + suffix)
        !jupyter nbconvert --to html --execute display_nonmatches.ipynb --output $filename
        
    except Exception as e:
        print("--------  ERROR: ", str(e))
        print()
        continue

    print("END processing dataset: ", key)
